In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, roc_auc_score, f1_score,
                             confusion_matrix, classification_report,
                             precision_score, recall_score)
import warnings
warnings.filterwarnings("ignore")
 
DATA_DIR = Path("../DATA")

### STEP 1. 데이터 로드

- sensor_hrv  : 79,639행 × 30열
- sleep_diary : 1,372행 × 11열
- survey      : 49행 × 23열

In [ ]:
# ================================================================
# STEP 1. 데이터 로드
# ================================================================
print("=" * 60)
print("STEP 1. 데이터 로드")
print("=" * 60)
 
hrv    = pd.read_csv(DATA_DIR / "sensor_hrv.csv")
sleep  = pd.read_csv(DATA_DIR / "sleep_diary.csv")
survey = pd.read_csv(DATA_DIR / "survey.csv")
 
# ts_start: 밀리초 Unix timestamp → datetime 변환
hrv["ts_start"] = pd.to_datetime(hrv["ts_start"], unit="ms")
hrv["ts_end"]   = pd.to_datetime(hrv["ts_end"],   unit="ms")
hrv["hour"]     = hrv["ts_start"].dt.hour
hrv["date"]     = hrv["ts_start"].dt.date
 
sleep["date"]   = pd.to_datetime(sleep["date"]).dt.date
 
print(f"  sensor_hrv  : {hrv.shape[0]:,}행 × {hrv.shape[1]}열")
print(f"  sleep_diary : {sleep.shape[0]:,}행 × {sleep.shape[1]}열")
print(f"  survey      : {survey.shape[0]:,}행 × {survey.shape[1]}열\n")

### STEP 2. 라벨 생성
- 기존 방식 : phq-9 설문 점수(0~27)를 기준으로 우울 증상을 4단계로 구분함
- 0 ~ 4점까지는 우울이 아니라고 판단하고 그 이상부터를 경미한 우울부터 심각한 우울증으로 판단함
- 따라서 phq-9 설문 점수를 기준으로 0~4는 0, 그 이상을 1로 라벨을 생성함

- 라벨 수: 98
- 분포 → 0(정상): 63명, 1(우울): 35명

In [ ]:
# ================================================================
# STEP 2. 라벨 생성
#   PHQ9_1 → period=1, PHQ9_2 → period=2
#   점수 <= 4 → 0 (정상), > 4 → 1 (우울 위험)
# ================================================================
print("=" * 60)
print("STEP 2. 라벨 생성")
print("=" * 60)
 
records = []
for _, row in survey.iterrows():
    for period, col in [(1, "PHQ9_1"), (2, "PHQ9_2")]:
        score = row.get(col, np.nan)
        if pd.isna(score):
            continue
        records.append({
            "deviceId" : row["deviceId"],
            "period"   : period,
            "phq9"     : score,
            "label"    : 0 if score <= 4 else 1
        })
 
labels = pd.DataFrame(records)
print(f"  라벨 수: {len(labels)}")
print(f"  분포 → 0(정상): {(labels['label']==0).sum()}명, "
      f"1(우울): {(labels['label']==1).sum()}명\n")

### STEP 3. 조사 기간 확인 및 슬라이싱 함수

In [ ]:
# HRV 데이터 확인
hrv_check = hrv.groupby("deviceId").agg(
    행수       = ("ts_start", "count"),
    시작일     = ("date", "min"),
    종료일     = ("date", "max"),
    총일수     = ("date", "nunique")
).reset_index()

hrv_check["기간(일)"] = (hrv_check["종료일"] - hrv_check["시작일"]).apply(lambda x: x.days + 1)
print("=== HRV 피험자별 데이터 현황 ===")
print(hrv_check.to_string(index=False))

print(f"\n총 피험자 수: {len(hrv_check)}")
print(f"총일수 14일 미만: {(hrv_check['총일수'] < 14).sum()}명")
print(f"총일수 14~28일:   {(hrv_check['총일수'].between(14,28)).sum()}명")
print(f"총일수 28일 초과: {(hrv_check['총일수'] > 28).sum()}명")

```
=== HRV 피험자별 데이터 현황 ===
deviceId   행수        시작일        종료일  총일수  기간(일)
    0000 2028 0000-00-00 0000-00-00   28     32
    0000  725 0000-00-00 0000-00-00   32     33
    0000  809 0000-00-00 0000-00-00   27     30
    0000 2209 0000-00-00 0000-00-00   30     36
    0000  833 0000-00-00 0000-00-00   28     28
```

- 기간의 절반에 해당하는 일자를 파악해 반으로 나눔

In [ ]:
def get_hrv_2weeks(device_id: str, period: int) -> pd.DataFrame:
    sub = hrv[hrv["deviceId"] == device_id].copy()
    if sub.empty:
        return sub
    sub = sub.sort_values("ts_start")
    
    dates = sorted(sub["date"].unique())
    mid_date = dates[len(dates) // 2]  # 날짜 목록 정중앙
    
    if period == 1:
        return sub[sub["date"] < mid_date]
    else:
        return sub[sub["date"] >= mid_date]

def get_sleep_2weeks(device_id: str, period: int) -> pd.DataFrame:
    sub = sleep[sleep["deviceId"] == device_id].copy()
    if sub.empty:
        return sub
    sub = sub.sort_values("date")
    
    dates = sorted(sub["date"].unique())
    mid_date = dates[len(dates) // 2]
    
    if period == 1:
        return sub[sub["date"] < mid_date]
    else:
        return sub[sub["date"] >= mid_date]

In [ ]:
result = []
for did in hrv["deviceId"].unique():
    sub = hrv[hrv["deviceId"] == did].copy()
    sub = sub.sort_values("ts_start")
    
    dates = sorted(sub["date"].unique())
    mid_idx = len(dates) // 2
    mid_date = dates[mid_idx]  # 날짜 목록의 정중앙 날짜
    
    p1 = sub[sub["date"] < mid_date]
    p2 = sub[sub["date"] >= mid_date]
    
    result.append({
        "deviceId"  : did,
        "시작일"    : dates[0],
        "중간일"    : mid_date,
        "종료일"    : dates[-1],
        "p1_days"   : p1["date"].nunique(),
        "p2_days"   : p2["date"].nunique(),
        "p1_rows"   : len(p1),
        "p2_rows"   : len(p2),
    })

check = pd.DataFrame(result)
print(check.to_string(index=False))
print(f"\np1 평균 일수: {check['p1_days'].mean():.1f}일")
print(f"p2 평균 일수: {check['p2_days'].mean():.1f}일")

### STEP 4. Feature 추출 함수

- [생체 feature]
-   전체 집계  : HR, ibi, rmssd, pnn50, lf/hf → agg, std
-   시간대 분할: HR, ibi, rmssd, pnn50, lf/hf
-                → day_agg, day_std, night_agg, night_std
-   ACC        : 전체 → agg, std / 야간 → night_std
-   추세       : HR, ibi, rmssd, pnn50, lf/hf → trend (후반-전반)
-
- [수면 feature]
-   전체 집계  : sleep_duration, sleep_latency, waso,
-                sleep_efficiency, wakeup@night → agg, std
-   불규칙성   : asleep_std, wakeup_std (시각 → 분 변환)
-   추세       : 위 5개 → trend
-
- [설문 feature]
-   sex, age

In [ ]:
MODE = "median"
AGG  = "median"
 
HRV_COLS   = ["HR", "ibi", "rmssd", "pnn50", "lf/hf"]
SLEEP_COLS = ["sleep_duration", "sleep_latency", "waso",
              "sleep_efficiency", "wakeup@night"]
 
 
def agg_col(series: pd.Series, method: str) -> float:
    """mean 또는 median 집계"""
    if method == "median":
        return series.median()
    return series.mean()
 
 
def time_to_minutes(t) -> float:
    """HH:MM(:SS) → 분, 새벽 시간 +1440 보정"""
    if pd.isna(t):
        return np.nan
    try:
        parts = str(t).split(":")
        m = int(parts[0]) * 60 + int(parts[1])
        return float(m + 1440 if m < 600 else m)
    except Exception:
        return np.nan
 
 
def extract_hrv_features(df: pd.DataFrame) -> dict:
    feats = {}
    if df.empty:
        return feats
 
    df = df.copy()
 
    # ACC magnitude
    df["acc_mag"] = np.sqrt(df["acc_x_avg"]**2 +
                            df["acc_y_avg"]**2 +
                            df["acc_z_avg"]**2)
 
    day   = df[df["hour"].between(9, 21)]
    night = df[~df["hour"].between(9, 21)]
 
    # 전체 집계: HRV 핵심 변수
    for col in HRV_COLS:
        if col not in df.columns:
            continue
        feats[f"{col}_{AGG}"] = agg_col(df[col].dropna(), AGG)
        feats[f"{col}_std"]   = df[col].std()
 
    # 시간대 분할: HRV 핵심 변수
    for col in HRV_COLS:
        if col not in df.columns:
            continue
        feats[f"{col}_day_{AGG}"]   = agg_col(day[col].dropna(), AGG)
        feats[f"{col}_day_std"]     = day[col].std()
        feats[f"{col}_night_{AGG}"] = agg_col(night[col].dropna(), AGG)
        feats[f"{col}_night_std"]   = night[col].std()
 
    # ACC 전체
    feats[f"acc_mag_{AGG}"] = agg_col(df["acc_mag"].dropna(), AGG)
    feats["acc_mag_std"]    = df["acc_mag"].std()
    # ACC 야간 뒤척임
    feats["acc_mag_night_std"] = night["acc_mag"].std()
 
    # 추세: 전반 vs 후반
    if "day_rank" in df.columns:
        w1 = df[df["day_rank"] <= (df["day_rank"].max() // 2)]
        w2 = df[df["day_rank"] >  (df["day_rank"].max() // 2)]
        for col in HRV_COLS:
            if col in df.columns:
                feats[f"{col}_trend"] = (agg_col(w2[col].dropna(), AGG) -
                                         agg_col(w1[col].dropna(), AGG))
 
    return feats
 
 
def extract_sleep_features(df: pd.DataFrame) -> dict:
    feats = {}
    if df.empty:
        return feats
 
    # 전체 집계
    for col in SLEEP_COLS:
        if col not in df.columns:
            continue
        feats[f"{col}_{AGG}"] = agg_col(df[col].dropna(), AGG)
        feats[f"{col}_std"]   = df[col].std()
 
    # 취침/기상 시간 불규칙성
    for col, label in [("asleep", "asleep"), ("wakeup", "wakeup")]:
        if col in df.columns:
            mins = df[col].apply(time_to_minutes)
            feats[f"{label}_std"] = mins.std()
 
    # 추세
    if len(df) >= 2:
        mid = len(df) // 2
        w1  = df.iloc[:mid]
        w2  = df.iloc[mid:]
        for col in SLEEP_COLS:
            if col in df.columns:
                feats[f"{col}_trend"] = (agg_col(w2[col].dropna(), AGG) -
                                         agg_col(w1[col].dropna(), AGG))
 
    return feats
 
 
def extract_survey_features(row: pd.Series) -> dict:
    feats = {}
    for col in ["sex", "age"]:
        if col in row.index:
            feats[col] = row[col]
    return feats

### STEP 5. Feature 테이블 생성

In [ ]:
print("=" * 60)
print("STEP 5. Feature 테이블 생성")
print("=" * 60)
 
rows    = []
skipped = 0
 
for _, lrow in labels.iterrows():
    did    = lrow["deviceId"]
    period = lrow["period"]
 
    hrv_h   = get_hrv_2weeks(did, period)
    sleep_h = get_sleep_2weeks(did, period)
 
    if hrv_h.empty:
        print(f"  경고: {did} period={period} 생체 데이터 없음 → 스킵")
        skipped += 1
        continue
 
    # day_rank 추가 (추세 계산용)
    hrv_h = hrv_h.copy()
    min_date = hrv_h["date"].min()
    hrv_h["day_rank"] = hrv_h["date"].apply(lambda d: (d - min_date).days + 1)
 
    survey_row  = survey[survey["deviceId"] == did]
    survey_feat = extract_survey_features(survey_row.iloc[0]) \
                  if not survey_row.empty else {}
 
    rows.append({
        "deviceId" : did,
        "period"   : period,
        **extract_hrv_features(hrv_h),
        **extract_sleep_features(sleep_h),
        **survey_feat,
        "label"    : lrow["label"],
        "phq9"     : lrow["phq9"],
    })
 
feat_df = pd.DataFrame(rows)
print(f"  스킵: {skipped}개")
print(f"  완성 테이블: {feat_df.shape[0]}행 × {feat_df.shape[1]}열")
print(f"  label 분포 → 0: {(feat_df['label']==0).sum()}, "
      f"1: {(feat_df['label']==1).sum()}\n")
 
DROP   = ["deviceId", "period", "phq9", "label"]
f_cols = [c for c in feat_df.columns if c not in DROP]
print(f"  생성된 feature ({len(f_cols)}개):")
for i, c in enumerate(f_cols, 1):
    print(f"    {i:>2}. {c}")
 
feat_df.to_csv(f"features_{MODE}.csv", index=False)
print(f"\n  features_{MODE}.csv 저장 완료!\n")

STEP 5. Feature 테이블 생성

-  스킵: 0개
-  완성 테이블: 98행 × 61열
-  label 분포 → 0: 63, 1: 35

-  생성된 feature (57개):
     1. HR_median
     2. HR_std
     3. ibi_median
     4. ibi_std
     5. rmssd_median
     6. rmssd_std
     7. pnn50_median
     8. pnn50_std
     9. lf/hf_median
    10. lf/hf_std
    11. HR_day_median
    12. HR_day_std
    13. HR_night_median
    14. HR_night_std
    15. ibi_day_median
    16. ibi_day_std
    17. ibi_night_median
...

### STEP 6. 전처리

In [ ]:
print("=" * 60)
print("STEP 6. 전처리")
print("=" * 60)
 
X       = feat_df[f_cols].copy()
y       = feat_df["label"].values
devices = feat_df["deviceId"].values
 
missing_rate = X.isnull().mean().sort_values(ascending=False)
print("  결측률 상위 10개:")
print(missing_rate.head(10).to_string())
print()
 
# 결측률 40% 초과 제거
drop_cols = missing_rate[missing_rate > 0.4].index.tolist()
if drop_cols:
    print(f"  결측률 40% 초과 제거: {drop_cols}")
    X      = X.drop(columns=drop_cols)
    f_cols = [c for c in f_cols if c not in drop_cols]
 
# 나머지 결측 → 중앙값 대체
for col in X.columns:
    if X[col].isnull().any():
        X[col] = X[col].fillna(X[col].median())
 
print(f"\n  최종 feature 수: {len(f_cols)}개")
print(f"  남은 결측치: {X.isnull().sum().sum()}개\n")

- 결측률 상위 10개:
    - HR_median       0.0
    - HR_std          0.0
    - ibi_median      0.0
    - ibi_std         0.0
    - rmssd_median    0.0
    - rmssd_std       0.0
    - pnn50_median    0.0
    - pnn50_std       0.0
    - lf/hf_median    0.0
    - lf/hf_std       0.0

### STEP 7. RandomForest + LOSO-CV

print("=" * 60)
print("STEP 7. LOSO-CV (Random Forest)")
print("=" * 60)
 
unique_devices = np.unique(devices)
all_probs  = []
all_labels = []
 
for test_device in unique_devices:
    train_idx = np.where(devices != test_device)[0]
    test_idx  = np.where(devices == test_device)[0]
 
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
 
    scaler     = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train)
    X_test_sc  = scaler.transform(X_test)
 
    model = RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )
    model.fit(X_train_sc, y_train)
    all_probs.extend(model.predict_proba(X_test_sc)[:, 1])
    all_labels.extend(y_test)
 
all_probs  = np.array(all_probs)
all_labels = np.array(all_labels)
 
auc = roc_auc_score(all_labels, all_probs)
print(f"  AUC-ROC : {auc:.3f}\n")
print(f"  {'threshold':>10}  {'Accuracy':>8}  {'Precision':>9}  "
      f"{'Recall':>7}  {'F1':>6}")
print("  " + "-" * 50)
 
for thr in [0.50, 0.45, 0.40, 0.35, 0.30]:
    p = (all_probs >= thr).astype(int)
    print(f"  {thr:>10.2f}"
          f"  {accuracy_score(all_labels, p):>8.3f}"
          f"  {precision_score(all_labels, p, zero_division=0):>9.3f}"
          f"  {recall_score(all_labels, p, zero_division=0):>7.3f}"
          f"  {f1_score(all_labels, p, zero_division=0):>6.3f}")
 
BEST_THR  = 0.4
all_preds = (all_probs >= BEST_THR).astype(int)
cm        = confusion_matrix(all_labels, all_preds)
 
print(f"\n  → threshold={BEST_THR} 상세 결과")
print("=" * 60)
print(f"  Accuracy : {accuracy_score(all_labels, all_preds):.3f}")
print(f"  AUC-ROC  : {auc:.3f}")
print(f"  F1-score : {f1_score(all_labels, all_preds):.3f}")
print()
print("  Confusion Matrix:")
print(f"              예측 0   예측 1")
print(f"  실제 0  :   {cm[0,0]:>4}     {cm[0,1]:>4}")
print(f"  실제 1  :   {cm[1,0]:>4}     {cm[1,1]:>4}")
print()
print(classification_report(all_labels, all_preds,
                            target_names=["PHQ<=4 (0)", "PHQ>4 (1)"]))

```
============================================================
STEP 7. LOSO-CV (Random Forest)
============================================================
  AUC-ROC : 0.678

   threshold  Accuracy  Precision   Recall      F1
  --------------------------------------------------
        0.50     0.673      0.600    0.257   0.360
        0.45     0.673      0.556    0.429   0.484
        0.40     0.684      0.553    0.600   0.575
        0.35     0.612      0.469    0.657   0.548
        0.30     0.561      0.435    0.771   0.557

  → threshold=0.4 상세 결과
============================================================
  Accuracy : 0.684
  AUC-ROC  : 0.678
  F1-score : 0.575

  Confusion Matrix:
              예측 0   예측 1
  실제 0  :     46       17
  실제 1  :     14       21

              precision    recall  f1-score   support
...
    accuracy                           0.68        98
   macro avg       0.66      0.67      0.66        98
weighted avg       0.69      0.68      0.69        98

```

### STEP 8. Feature Importance

In [ ]:
print("=" * 60)
print("STEP 8. Feature Importance")
print("=" * 60)
 
scaler_full = StandardScaler()
X_full_sc   = scaler_full.fit_transform(X)
 
model_full = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)
model_full.fit(X_full_sc, y)
 
imp_df = pd.DataFrame({
    "feature"    : f_cols,
    "importance" : model_full.feature_importances_
}).sort_values("importance", ascending=False).reset_index(drop=True)
 
print(imp_df.head(20).to_string(index=False))
imp_df.to_csv(f"feature_importance_{MODE}.csv", index=False)
print(f"\n  feature_importance_{MODE}.csv 저장 완료!")

- feature  importance
- sleep_duration_std    0.082458
- asleep_std    0.046927
- age    0.043135
- HR_std    0.034686
- rmssd_night_std    0.032994
- waso_std    0.032097
- rmssd_std    0.031261
- lf/hf_median    0.030650
- acc_mag_median    0.022058
- pnn50_night_std    0.021850
- lf/hf_night_std    0.021776
- sleep_latency_trend    0.019789
- HR_trend    0.018505
- waso_median    0.018194
- pnn50_trend    0.018116
- HR_day_std    0.017872
- lf/hf_std    0.017282
- lf/hf_night_median    0.016957
- ibi_day_median    0.016944
- lf/hf_day_median    0.016803

In [ ]:
from sklearn.linear_model import LogisticRegression

imp_df = pd.DataFrame({
    "feature"    : f_cols,
    "importance" : model_full.feature_importances_
}).sort_values("importance", ascending=False).reset_index(drop=True)

results = []
BEST_THR = 0.50

for top_n in [10, 20]:
    top_features = imp_df["feature"].head(top_n).tolist()
    X_top        = X[top_features]

    all_probs  = []
    all_labels = []

    for test_device in unique_devices:
        train_idx = np.where(devices != test_device)[0]
        test_idx  = np.where(devices == test_device)[0]

        X_train, X_test = X_top.iloc[train_idx], X_top.iloc[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        scaler     = StandardScaler()
        X_train_sc = scaler.fit_transform(X_train)
        X_test_sc  = scaler.transform(X_test)

        model_lr = LogisticRegression(
            class_weight="balanced",
            max_iter=1000,
            random_state=42
        )
        model_lr.fit(X_train_sc, y_train)
        all_probs.extend(model_lr.predict_proba(X_test_sc)[:, 1])
        all_labels.extend(y_test)

    all_probs  = np.array(all_probs)
    all_labels = np.array(all_labels)
    auc        = roc_auc_score(all_labels, all_probs)

    print(f"\n{'='*55}")
    print(f"  Logistic Regression - 상위 {top_n}개 feature")
    print(f"{'='*55}")
    print(f"  사용 변수: {top_features}\n")
    print(f"  AUC-ROC : {auc:.3f}\n")
    print(f"  {'threshold':>10}  {'Accuracy':>8}  {'Precision':>9}  "
          f"{'Recall':>7}  {'F1':>6}")
    print("  " + "-" * 50)

    for thr in [0.50, 0.45, 0.40, 0.35, 0.30]:
        p = (all_probs >= thr).astype(int)
        print(f"  {thr:>10.2f}"
              f"  {accuracy_score(all_labels, p):>8.3f}"
              f"  {precision_score(all_labels, p, zero_division=0):>9.3f}"
              f"  {recall_score(all_labels, p, zero_division=0):>7.3f}"
              f"  {f1_score(all_labels, p, zero_division=0):>6.3f}")

    p_best = (all_probs >= BEST_THR).astype(int)
    results.append({
        "top_n"     : top_n,
        "auc"       : auc,
        "accuracy"  : accuracy_score(all_labels, p_best),
        "precision" : precision_score(all_labels, p_best, zero_division=0),
        "recall"    : recall_score(all_labels, p_best, zero_division=0),
        "f1"        : f1_score(all_labels, p_best, zero_division=0),
    })

print(f"\n{'='*70}")
print(f"  최종 비교 (threshold={BEST_THR} 기준)")
print(f"{'='*70}")
print(f"  {'모델':<35} {'AUC':>6}  {'Accuracy':>8}  {'Recall':>7}  {'F1':>6}")
print(f"  {'-'*63}")
print(f"  {'Random Forest (전체 feature)':<35} {'0.678':>6}  {'0.684':>8}  {'0.600':>7}  {'0.575':>6}")
for r in results:
    label = f"Logistic Regression (상위 {r['top_n']}개)"
    print(f"  {label:<35} {r['auc']:>6.3f}  {r['accuracy']:>8.3f}  {r['recall']:>7.3f}  {r['f1']:>6.3f}")

In [ ]:
'''
=======================================================
  Logistic Regression - 상위 10개 feature
=======================================================
  사용 변수: ['sleep_duration_std', 'asleep_std', 'age', 'HR_std', 'rmssd_night_std', 'waso_std', 'rmssd_std', 'lf/hf_median', 'acc_mag_median', 'pnn50_night_std']

  AUC-ROC : 0.787

   threshold  Accuracy  Precision   Recall      F1
  --------------------------------------------------
        0.50     0.724      0.587    0.771   0.667
        0.45     0.714      0.571    0.800   0.667
        0.40     0.684      0.538    0.800   0.644
        0.35     0.653      0.508    0.857   0.638
        0.30     0.643      0.500    0.857   0.632

=======================================================
  Logistic Regression - 상위 20개 feature
=======================================================
  사용 변수: ['sleep_duration_std', 'asleep_std', 'age', 'HR_std', 'rmssd_night_std', 'waso_std', 'rmssd_std', 'lf/hf_median', 'acc_mag_median', 'pnn50_night_std', 'lf/hf_night_std', 'sleep_latency_trend', 'HR_trend', 'waso_median', 'pnn50_trend', 'HR_day_std', 'lf/hf_std', 'lf/hf_night_median', 'ibi_day_median', 'lf/hf_day_median']

  AUC-ROC : 0.713

   threshold  Accuracy  Precision   Recall      F1
  --------------------------------------------------
...
  ---------------------------------------------------------------
  Random Forest (전체 feature)           0.678     0.684    0.600   0.575
  Logistic Regression (상위 10개)         0.787     0.724    0.771   0.667
  Logistic Regression (상위 20개)         0.713     0.673    0.629   0.579
'''